# 버그 찾기 2 (쉬움~보통): 엄격한 검정까지 통과했는데

위치 힌트: **분할부터 평가까지**의 셀들 중에 있다. 귀무분포 셀 자체에는 버그가 없다.

데이터: facts를 흉내 낸 **합성** 활성값이다. 사실 200개 × 바꿔 말하기 4개 = 800문장, d=256. **같은 사실을 바꿔 말한 문장은 라벨이 같다.** `group`이 사실 번호다.

**하는 법**
1. 위에서부터 끝까지 실행하고, 맨 아래 **결론**을 읽는다.
2. 코드나 설명 중 "말과 실제 동작이 다른 곳"을 찾아 **문장으로** 적는다(맨 아래 `내 답` 셀).
3. 고치면 숫자가 어떻게 바뀔지 **한 줄 예측**을 먼저 적는다.
4. 고쳐서 다시 돌린다. 문법이 막히면 고칠 내용을 문장으로 쓰고 `???`로 남겨 두면 Claude가 채운다.

이 노트북은 오류 없이 끝까지 돈다. 정답은 Claude에게 따로 있다.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

d = np.load("acts/p2.npz")
X, y, group = d["X"], d["y"], d["group"]
print(X.shape, "참 비율", y.mean(), "사실 수", len(np.unique(group)))

In [ ]:
# 층화 분할: train과 test의 참/거짓 비율을 똑같이 맞춘다 → 클래스 비율 때문에 생기는 착시가 없다
idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.25, stratify=y, random_state=0)
print("train 참 비율", y[tr].mean(), "test 참 비율", y[te].mean())

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=5000).fit(X[tr], y[tr])
observed = roc_auc_score(y[te], lr.decision_function(X[te]))
print(f"test AUROC: {observed:.3f}")

## 순열 귀무분포

train 라벨만 섞어서 다시 학습하고, 섞지 않은 test에서 AUROC를 잰다(probe_lab.permutation_null과 같다). 묻는 질문: "라벨이 활성값과 무관하다면, 이 절차가 이만한 AUROC를 얼마나 자주 우연히 내는가?"

In [ ]:
rng = np.random.default_rng(0)
null = []
for _ in range(50):
    probe = LogisticRegression(C=1.0, max_iter=5000).fit(X[tr], rng.permutation(y[tr]))
    null.append(roc_auc_score(y[te], probe.decision_function(X[te])))
null = np.array(null)
p = (1 + np.sum(null >= observed)) / (1 + len(null))
print(f"null: {null.mean():.3f} ± {null.std():.3f}   p = {p:.3f}")

## 결론

1. test AUROC 0.85, 귀무분포 0.50 ± 0.05, p ≈ 0.02(50번 섞기로 낼 수 있는 가장 작은 값). 우연이 아니다.
2. 층화 분할로 클래스 비율 착시도 없앴으니 공정한 평가다.
3. 이 층에는 문장의 참/거짓을 읽어 내는 선형 방향이 있다.

## 내 답

- 말과 실제 동작이 다른 곳:
- 그래서 틀린 결론 문장(번호):
- 고치면 어떻게 바뀔지 예측:
- 고친 뒤 실제 결과: